# Plant placement optimizer (Module 3)

This notebook is Module 3 of the Seat-Level Visibility Toolkit. It answers one design question: **where should the next plants go so that the seats that see the least green gain the most?**

It uses the same room files and the same calculation engine as the Module 1 notebook (`objects.csv` and `seats.csv`). It tries many candidate positions for a new plant, re-scores every seat each time, and adds the best one, round after round (a *greedy* search).

**How to use it:** run the cells from top to bottom (▶). The only things to choose are in Step 1 and are dropdowns or boxes, so there is no code to edit.

**Which pass rule it optimizes:** a seat passes when it has a line of sight to a plant (and, if you set it, a minimum GVI). In the WELL v2 Q2 2025 text, a seat also passes M09 Part 1 if it is within 33 ft of a plant, and in a small room that alternative may already pass every seat. Step 4b checks this first. The search is still useful when a design aims at a line of sight for every seat, in larger rooms, and for raising the green that the seats see.

**Before you rely on any result**
- It optimizes **visible green only**. It does not judge species, light, watering, maintenance, allergies or looks, and Terrapin's guide puts quality and diversity above quantity. A designer must review every suggestion.
- Plants are modeled as spheres, so GVI is approximate. All defaults (eye height 1.2 m, 60° × 60° view, "any one clear ray") are project assumptions, not values from WELL or LEED.
- Circulation is only approximated by a clearance around seats and a keep-out box you can draw. Fire routes, doors and accessibility must be checked by a person.
- This is a research tool, not an official WELL or LEED checker.

## Step 1: Settings

**Scoring settings** (same meaning as in Module 1; keep them identical to the Module 1 run you are improving):

`eye_height`, `fov_h`, `fov_v`, `n_h`, `n_v`, `min_clear_rays`, `min_gvi_pct`, `threshold_pct`.

**Optimizer settings** are the form fields in the second cell:

| Field | Meaning |
|---|---|
| `GOAL` | *reach 75% of seats*: fewest plants until the share of passing seats meets the threshold (stops early). *lift the weakest seats*: use all plants to raise the seats with the least green first. *raise the average*: use all plants to raise the mean GVI. |
| `PLANT_SIZE` | Size from a small catalog: small = 0.75 m tall, medium = 1.0 m (like the palms in the model), large = 1.25 m. |
| `MAX_PLANTS` | Plant budget (the most plants it may add). |
| `GRID_STEP` | Spacing of candidate positions in meters. Smaller is finer but slower. |
| `SEAT_CLEARANCE` | No plant closer than this to a seat (meters), so plants do not block the seat or its view. |
| `PLANT_GAP` | Smallest gap between two plants (meters). |
| `PUT_ON_TOP_OF` | Words from object names whose **top surface** may hold a plant, separated by commas (default: cabinets and the plant table). Leave empty for floor only. |
| `KEEP_OUT` | Optional boxes where no plant may go, written as `x0,y0,x1,y1` in meters and separated by `;` (for example a door swing). |

In [ ]:
params = dict(
    eye_height=1.2,
    fov_h=60,
    fov_v=60,
    n_h=60,
    n_v=60,
    min_clear_rays=1,
    min_gvi_pct=0.0,
    threshold_pct=75,
)
params

In [ ]:
GOAL = "reach 75% of seats"  #@param ["reach 75% of seats", "lift the weakest seats", "raise the average"]
PLANT_SIZE = "medium"  #@param ["small", "medium", "large"]
MAX_PLANTS = 6  #@param {type:"integer"}
GRID_STEP = 0.5  #@param {type:"number"}
SEAT_CLEARANCE = 0.8  #@param {type:"number"}
PLANT_GAP = 0.15  #@param {type:"number"}
PUT_ON_TOP_OF = "cabinet, plant_table"  #@param {type:"string"}
KEEP_OUT = ""  #@param {type:"string"}

SIZES = {"small": 0.25, "medium": 0.375, "large": 0.5}   # sphere radius (m); plant height = 0.25 + 2 r
R_NEW = SIZES[PLANT_SIZE]
print(f"New plant: {PLANT_SIZE}, radius {R_NEW} m, about {0.25 + 2 * R_NEW:.2f} m tall; goal: {GOAL}; budget {MAX_PLANTS}")

## Step 2: The calculation engine (no need to edit)

This is the same engine as Module 1 (rays for GVI, sample points for the line-of-sight test). See the Module 1 notebook for the definitions.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Wedge

# ---------- objects ----------
def box(name, mn, mx, tag='solid'):
    return dict(name=name, type='box', tag=tag, mn=np.array(mn, float), mx=np.array(mx, float))

def sphere(name, c, r, tag='vegetation'):
    return dict(name=name, type='sphere', tag=tag, c=np.array(c, float), r=float(r))

def hit_t(o, D, obj):
    """Distance from origin o along each unit ray (rows of D) to the object; inf if the ray misses."""
    if obj['type'] == 'box':
        D = np.where(np.abs(D) < 1e-12, 1e-12, D)
        t1 = (obj['mn'] - o) / D
        t2 = (obj['mx'] - o) / D
        tmin = np.minimum(t1, t2).max(axis=1)
        tmax = np.maximum(t1, t2).min(axis=1)
        ok = (tmax >= np.maximum(tmin, 0)) & (tmax > 1e-9)
        t = np.where(tmin > 1e-9, tmin, tmax)
        return np.where(ok, t, np.inf)
    oc = o - obj['c']
    b = D @ oc
    c = oc @ oc - obj['r'] ** 2
    disc = b * b - c
    sq = np.sqrt(np.maximum(disc, 0))
    t = -b - sq
    t = np.where(t > 1e-9, t, -b + sq)
    ok = (disc >= 0) & (t > 1e-9)
    return np.where(ok, t, np.inf)

# ---------- view geometry ----------
def frame(yaw_deg):
    y = math.radians(yaw_deg)
    f = np.array([math.cos(y), math.sin(y), 0.0])      # forward
    r = np.array([math.sin(y), -math.cos(y), 0.0])     # right
    u = np.array([0.0, 0.0, 1.0])                      # up
    return f, r, u

def view_rays(yaw_deg, P):
    """Unit ray directions through an evenly spaced grid on a flat view plane."""
    f, r, u = frame(yaw_deg)
    a = ((np.arange(P['n_h']) + 0.5) / P['n_h'] * 2 - 1) * math.tan(math.radians(P['fov_h'] / 2))
    b = ((np.arange(P['n_v']) + 0.5) / P['n_v'] * 2 - 1) * math.tan(math.radians(P['fov_v'] / 2))
    A, B = np.meshgrid(a, b)
    D = f + A[..., None] * r + B[..., None] * u
    D = D.reshape(-1, 3)
    return D / np.linalg.norm(D, axis=1, keepdims=True), (P['n_v'], P['n_h'])

def gvi_at(o, yaw_deg, objects, P):
    """GVI (%) and the plant/not-plant grid for an eye point o facing yaw_deg."""
    D, shape = view_rays(yaw_deg, P)
    ts = np.stack([np.full(len(D), np.inf) if ob['tag'] == 'glass' else hit_t(o, D, ob)
                   for ob in objects], axis=1)
    idx = ts.argmin(axis=1)
    finite = np.isfinite(ts.min(axis=1))
    tags = np.array([ob['tag'] for ob in objects] + ['none'])
    veg = np.isin(tags[np.where(finite, idx, len(objects))], ['vegetation', 'exterior'])
    return float(veg.mean() * 100), veg.reshape(shape)

# ---------- line of sight ----------
def surface_samples(obj, n=40):
    """Sample points on an object's surface, with outward normals."""
    if obj['type'] == 'sphere':
        i = np.arange(n) + 0.5
        phi = np.arccos(1 - 2 * i / n)
        th = math.pi * (1 + 5 ** 0.5) * i
        N = np.stack([np.cos(th) * np.sin(phi), np.sin(th) * np.sin(phi), np.cos(phi)], axis=1)
        return obj['c'] + obj['r'] * N, N
    g = np.linspace(0.05, 0.95, 5)
    pts, nor = [], []
    for k in range(3):
        i, j = [m for m in range(3) if m != k]
        for side, sgn in ((obj['mn'][k], -1.0), (obj['mx'][k], 1.0)):
            for gi in g:
                for gj in g:
                    p = np.zeros(3)
                    p[k] = side
                    p[i] = obj['mn'][i] + gi * (obj['mx'][i] - obj['mn'][i])
                    p[j] = obj['mn'][j] + gj * (obj['mx'][j] - obj['mn'][j])
                    nv = np.zeros(3); nv[k] = sgn
                    pts.append(p); nor.append(nv)
    return np.array(pts), np.array(nor)

def los_check(o, yaw_deg, objects, P, tag='vegetation'):
    """Returns (clear sample points, plants with a clear point, plants inside the view angle)."""
    f, r, u = frame(yaw_deg)
    tan_h = math.tan(math.radians(P['fov_h'] / 2))
    tan_v = math.tan(math.radians(P['fov_v'] / 2))
    solids = [ob for ob in objects if ob['tag'] == 'solid']
    total, seen, inview = 0, [], []
    for ob in objects:
        if ob['tag'] != tag:
            continue
        pts, nor = surface_samples(ob)
        v = pts - o
        facing = np.einsum('ij,ij->i', nor, -v) > 0           # surface faces the seat
        fwd, right, up = v @ f, v @ r, v @ u
        inside = facing & (fwd > 1e-9) & (np.abs(right) <= tan_h * fwd) & (np.abs(up) <= tan_v * fwd)
        if not inside.any():
            continue
        inview.append(ob['name'])
        q = pts[inside]
        dist = np.linalg.norm(q - o, axis=1)
        D = (q - o) / dist[:, None]
        blocked = np.zeros(len(q), bool)
        for s in solids:
            blocked |= hit_t(o, D, s) < dist - 1e-6
        n_clear = int((~blocked).sum())
        if n_clear:
            seen.append(ob['name'])
            total += n_clear
    return total, seen, inview

# ---------- audit ----------
def audit(seats, objects, P):
    rows = []
    for s in seats:
        eh = s.get('eye_height', P['eye_height'])
        o = np.array([s['x'], s['y'], eh], float)
        gvi, _ = gvi_at(o, s['facing_deg'], objects, P)
        clear, seen, inview = los_check(o, s['facing_deg'], objects, P)
        ok = (clear >= P['min_clear_rays']) and (gvi >= P['min_gvi_pct'])
        rows.append(dict(seat_id=s['id'], zone=s.get('zone', 'all'), x=s['x'], y=s['y'],
                         facing_deg=s['facing_deg'], plants_in_view=len(inview),
                         plants_clear=len(seen), clear_points=clear,
                         gvi_pct=round(gvi, 1), los_pass=bool(ok)))
    return pd.DataFrame(rows)

def share_passing(df, P):
    n = len(df)
    need = math.ceil(P['threshold_pct'] / 100 * n)
    passed = int(df.los_pass.sum())
    return dict(seats=n, passed=passed, share_pct=round(passed / n * 100, 1) if n else float('nan'),
                needed=need, meets=bool(passed >= need) if n else False)

## Step 3: Load the room

Use the same two files as in Module 1. Tick `USE_UPLOAD`, run the cell, and choose `objects.csv` and `seats.csv` (Colab only sees files uploaded in this session). If you leave it unticked, a small toy room is used so you can see how the notebook works.

In [ ]:
USE_UPLOAD = False  #@param {type:"boolean"}

if USE_UPLOAD:
    from google.colab import files
    up = files.upload()
    seats_df = pd.read_csv('seats.csv')
    obj_df = pd.read_csv('objects.csv')
    seats = [{k: v for k, v in r_.items() if pd.notna(v)} for r_ in seats_df.to_dict('records')]
    objects = []
    for r_ in obj_df.to_dict('records'):
        if r_['type'] == 'box':
            objects.append(box(r_['name'], [r_['xmin'], r_['ymin'], r_['zmin']], [r_['xmax'], r_['ymax'], r_['zmax']], r_['tag']))
        else:
            objects.append(sphere(r_['name'], [r_['cx'], r_['cy'], r_['cz']], r_['r'], r_['tag']))
else:
    objects = [
        box('floor',      (-0.2, -0.2, -0.2), (8.2, 5.2, 0.0)),
        box('ceiling',    (-0.2, -0.2, 2.8),  (8.2, 5.2, 3.0)),
        box('west wall',  (-0.2, -0.2, 0.0),  (0.0, 5.2, 2.8)),
        box('south wall', (-0.2, -0.2, 0.0),  (8.2, 0.0, 2.8)),
        box('north wall', (-0.2, 5.0, 0.0),   (8.2, 5.2, 2.8)),
        # east wall with a window (y 1.0-4.0 m, z 0.9-2.4 m)
        box('east wall S',    (8.0, 0.0, 0.0), (8.2, 1.0, 2.8)),
        box('east wall N',    (8.0, 4.0, 0.0), (8.2, 5.0, 2.8)),
        box('east wall low',  (8.0, 1.0, 0.0), (8.2, 4.0, 0.9)),
        box('east wall high', (8.0, 1.0, 2.4), (8.2, 4.0, 2.8)),
        box('window glass',   (8.0, 1.0, 0.9), (8.2, 4.0, 2.4), 'glass'),
        # partition and plants
        box('partition', (3.0, 0.5, 0.0), (3.1, 2.6, 1.8)),
        sphere('plant 1', (4.5, 2.5, 0.9), 0.45),
        sphere('outdoor tree 1', (11.0, 2.0, 2.0), 1.5, 'exterior'),
        sphere('outdoor tree 2', (11.5, 3.8, 2.2), 1.6, 'exterior'),
    ]
    seats = [
        dict(id='A', x=1.0, y=3.5, facing_deg=0),
        dict(id='B', x=1.0, y=1.5, facing_deg=0),
        dict(id='C', x=5.8, y=2.5, facing_deg=0),
    ]
print(len(objects), 'objects,', len(seats), 'seats')

## Step 4: Today's situation (before any new plant)

Every seat is scored first with the plants already in the room.

In [ ]:
def plot_plan(df, objects, P, ax=None, cone_len=2.5, title=None, xlim=None, ylim=None):
    ax = ax or plt.subplots(figsize=(9, 6))[1]
    for ob in objects:
        if ob['type'] == 'box':
            if ob['mx'][2] <= 0.01 or ob['mn'][2] >= 2.0:
                continue                           # skip floor and ceiling in plan
            col = {'glass': '#9ec9f0', 'vegetation': '#7fc97f', 'exterior': '#7fc97f'}.get(ob['tag'], '#b8b8b0')
            ax.add_patch(Rectangle(ob['mn'][:2], *(ob['mx'][:2] - ob['mn'][:2]), color=col, alpha=0.9, lw=0))
        else:
            col = '#7fc97f' if ob['tag'] in ('vegetation','exterior') else '#b8b8b0'
            ax.add_patch(Circle(ob['c'][:2], ob['r'], color=col, alpha=0.7, lw=0))
    for _, s in df.iterrows():
        col = '#2ca02c' if s.los_pass else '#d62728'
        ax.add_patch(Wedge((s.x, s.y), cone_len, s.facing_deg - P['fov_h'] / 2, s.facing_deg + P['fov_h'] / 2,
                           color=col, alpha=0.15, lw=0))
        ax.plot(s.x, s.y, 'o', color='black', ms=6)
        ax.text(s.x, s.y + 0.25, f"{s.seat_id}: {'pass' if s.los_pass else 'fail'}, GVI {s.gvi_pct}%",
                ha='center', fontsize=9)
    ax.set_aspect('equal')
    if xlim is None or ylim is None:               # fit the picture to the room that was loaded
        pts = [(ob['mn'][0], ob['mn'][1], ob['mx'][0], ob['mx'][1]) for ob in objects
               if ob['type'] == 'box' and ob['mx'][2] > 0.01 and ob['mn'][2] < 2.0 and ob['tag'] != 'exterior']
        pts += [(ob['c'][0] - ob['r'], ob['c'][1] - ob['r'], ob['c'][0] + ob['r'], ob['c'][1] + ob['r']) for ob in objects if ob['type'] != 'box']
        x0, y0 = min(p[0] for p in pts), min(p[1] for p in pts)
        x1, y1 = max(p[2] for p in pts), max(p[3] for p in pts)
        xlim = xlim or (x0 - 0.5, x1 + 0.5); ylim = ylim or (y0 - 0.5, y1 + 0.5)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.set_title(title or 'Seat audit, plan view')
    return ax

In [ ]:
before = audit(seats, objects, params)
sb = share_passing(before, params)
print(f"Before: {sb['passed']}/{sb['seats']} seats pass ({sb['share_pct']}%); needed {sb['needed']} for {params['threshold_pct']}%")
print(f"GVI: min {before.gvi_pct.min():.1f}%, mean {before.gvi_pct.mean():.1f}%")
display(before[['seat_id', 'zone', 'gvi_pct', 'clear_points', 'los_pass']])

## Step 4b: Does the credit as written already pass?

WELL v2 (Q2 2025) M09 Part 1 also accepts a seat that is within 33 ft (10.06 m) of an indoor plant. This cell measures, for every seat, the straight distance from the eye to the nearest plant surface (walls ignored) and reports the share within the limit. If every seat is within the limit, no plant is needed for that alternative, and the placement search below answers the stricter question of a line of sight to a plant.

In [ ]:
WELL_DIST_FT = 33  #@param {type:"number"}
WELL_DIST_M = WELL_DIST_FT * 0.3048

def dist_to_object(o, ob):
    if ob['type'] == 'sphere':
        return max(float(np.linalg.norm(o - ob['c'])) - ob['r'], 0.0)
    d = np.maximum(np.maximum(ob['mn'] - o, 0), o - ob['mx'])
    return float(np.linalg.norm(d))

def distance_table(objs):
    rows = []
    for s in seats:
        o = np.array([s['x'], s['y'], s.get('eye_height', params['eye_height'])], float)
        dm = min([dist_to_object(o, ob) for ob in objs if ob['tag'] == 'vegetation'] or [float('inf')])
        rows.append(dict(seat_id=s['id'], dist_to_plant_m=round(dm, 2), within_limit=bool(dm <= WELL_DIST_M)))
    return pd.DataFrame(rows)

dist_before = distance_table(objects)
n_in = int(dist_before.within_limit.sum())
print(f"Within {WELL_DIST_FT} ft of a plant: {n_in} of {len(dist_before)} seats ({n_in / len(dist_before) * 100:.1f}%)")
if n_in / len(dist_before) * 100 >= params['threshold_pct']:
    print("The distance alternative is already met. The search below is for the stricter reading (line of sight to a plant) and for more green in view.")
display(dist_before)

## Step 5: Candidate positions

Candidates are points on a grid over the **floor** and over the **tops** of the objects named in `PUT_ON_TOP_OF`. A candidate is dropped if the plant would overlap a solid object, wall or window, come within `SEAT_CLEARANCE` of a seat, sit too close to another plant, or fall in a `KEEP_OUT` box.

In [ ]:
def parse_keepout(text):
    boxes = []
    for part in text.split(';'):
        v = [float(t) for t in part.replace(' ', '').split(',') if t != '']
        if len(v) == 4:
            boxes.append(v)
    return boxes

def make_candidates(objects, seats, r_new, step, clearance, gap, tops_words, keepout):
    boxes = [o for o in objects if o['type'] == 'box']
    solids = [o for o in boxes if o['tag'] in ('solid', 'glass')]
    plants = [o for o in objects if o['type'] == 'sphere' and o['tag'] == 'vegetation']
    floors = [o for o in boxes if o['tag'] == 'solid' and o['mx'][2] <= 0.01]
    if not floors:
        raise ValueError('No floor box found: add a thin box at z <= 0 that covers the room.')
    x0 = min(f['mn'][0] for f in floors); x1 = max(f['mx'][0] for f in floors)
    y0 = min(f['mn'][1] for f in floors); y1 = max(f['mx'][1] for f in floors)
    words = [w.strip().lower() for w in tops_words.split(',') if w.strip()]
    surfaces = [(0.0, (x0, y0, x1, y1), 'floor')]
    for b in solids:
        if b['tag'] == 'solid' and any(w in b['name'].lower() for w in words):
            surfaces.append((b['mx'][2], (b['mn'][0], b['mn'][1], b['mx'][0], b['mx'][1]), b['name']))
    cands, seen = [], set()
    for base, (ax0, ay0, ax1, ay1), on in surfaces:
        xs = np.arange(ax0 + r_new, ax1 - r_new + 1e-9, step) if on != 'floor' else np.arange(ax0, ax1 + 1e-9, step)
        ys = np.arange(ay0 + r_new, ay1 - r_new + 1e-9, step) if on != 'floor' else np.arange(ay0, ay1 + 1e-9, step)
        if on != 'floor' and len(xs) == 0: xs = np.array([(ax0 + ax1) / 2])
        if on != 'floor' and len(ys) == 0: ys = np.array([(ay0 + ay1) / 2])
        for x in xs:
            for y in ys:
                c = np.array([x, y, base + 0.25 + r_new])
                lo, hi = c - r_new, c + r_new
                if any(np.all(lo < b['mx'] - 1e-6) and np.all(hi > b['mn'] + 1e-6) for b in solids
                       if not (b['mx'][2] <= base + 1e-6 and on != 'floor' and b['name'] == on)):
                    continue                                   # overlaps a solid, wall or window
                if any(math.hypot(x - s['x'], y - s['y']) < clearance for s in seats):
                    continue
                if any(math.hypot(x - p['c'][0], y - p['c'][1]) < r_new + p['r'] + gap and abs(c[2] - p['c'][2]) < r_new + p['r'] + gap
                       for p in plants):
                    continue
                if any(a <= x <= c_ and b_ <= y <= d for a, b_, c_, d in keepout):
                    continue
                key = (round(x, 3), round(y, 3), round(base, 3))
                if key in seen:
                    continue
                seen.add(key)
                cands.append(dict(x=float(x), y=float(y), z=float(c[2]), on=on))
    return cands

cands = make_candidates(objects, seats, R_NEW, GRID_STEP, SEAT_CLEARANCE, PLANT_GAP, PUT_ON_TOP_OF, parse_keepout(KEEP_OUT))
n_floor = sum(c['on'] == 'floor' for c in cands)
print(f"{len(cands)} candidate positions ({n_floor} on the floor, {len(cands) - n_floor} on top of furniture)")
if not cands:
    raise ValueError('No candidate positions left: reduce SEAT_CLEARANCE or GRID_STEP, or empty KEEP_OUT.')

## Step 6: Search

**Speed:** instead of re-tracing every ray, the search remembers for each seat the nearest hit of every ray and only checks whether the new plant is nearer. The result is identical to a full Module 1 re-run, which Step 11 checks.

**How a position is chosen in each round**
- *reach 75% of seats:* the most seats passing, then the highest mean GVI;
- *lift the weakest seats:* the seat with the least green first (the lowest GVI is raised as far as possible, then the next lowest), then the most seats passing;
- *raise the average:* the highest mean GVI.

In [ ]:
solids_only = [ob for ob in objects if ob['tag'] == 'solid']

class SeatState:
    """What a seat currently sees, kept so a new plant can be tested quickly."""
    def __init__(self, s):
        self.o = np.array([s['x'], s['y'], s.get('eye_height', params['eye_height'])], float)
        self.yaw = s['facing_deg']
        self.D, _ = view_rays(self.yaw, params)
        ts = np.stack([np.full(len(self.D), np.inf) if ob['tag'] == 'glass' else hit_t(self.o, self.D, ob)
                       for ob in objects], axis=1)
        idx = ts.argmin(axis=1)
        self.tmin = ts.min(axis=1)
        tags = np.array([ob['tag'] for ob in objects] + ['none'])
        self.green = np.isin(tags[np.where(np.isfinite(self.tmin), idx, len(objects))], ['vegetation', 'exterior'])
        self.los = los_check(self.o, self.yaw, objects, params)[0]

states = [SeatState(s) for s in seats]

def new_plant(c, name='new plant'):
    return sphere(name, (c['x'], c['y'], c['z']), R_NEW)

# line-of-sight points to each candidate do not depend on other plants, so compute them once
cand_los = np.zeros((len(cands), len(states)), int)
cand_t = []
for i, c in enumerate(cands):
    sp = new_plant(c)
    cand_t.append([hit_t(st.o, st.D, sp) for st in states])
    for j, st in enumerate(states):
        cand_los[i, j] = los_check(st.o, st.yaw, solids_only + [sp], params)[0]

def evaluate(i):
    """GVI of every seat and the pass flags if candidate i were added now."""
    g, p = [], []
    for j, st in enumerate(states):
        gain = cand_t[i][j] < st.tmin
        gvi = float(np.where(gain, True, st.green).mean() * 100)
        g.append(gvi)
        p.append(bool((st.los + cand_los[i, j] >= params['min_clear_rays']) and gvi >= params['min_gvi_pct']))
    return np.array(g), np.array(p)

def current():
    g = np.array([st.green.mean() * 100 for st in states])
    p = np.array([(st.los >= params['min_clear_rays']) and gv >= params['min_gvi_pct'] for st, gv in zip(states, g)])
    return g, p

def score(g, p):
    if GOAL == "reach 75% of seats":
        return (int(p.sum()), float(g.mean()))
    if GOAL == "lift the weakest seats":
        return (tuple(np.round(np.sort(g), 4)), int(p.sum()))
    return (float(g.mean()), int(p.sum()))

need = math.ceil(params['threshold_pct'] / 100 * len(states))
chosen, log, first_round = [], [], None
g0, p0 = current()
for rnd in range(1, MAX_PLANTS + 1):
    if GOAL == "reach 75% of seats" and p0.sum() >= need:
        break
    free = [i for i in range(len(cands)) if i not in chosen and all(
        math.hypot(cands[i]['x'] - cands[k]['x'], cands[i]['y'] - cands[k]['y']) >= 2 * R_NEW + PLANT_GAP for k in chosen)]
    if not free:
        break
    evals = {i: evaluate(i) for i in free}
    best = max(free, key=lambda i: score(*evals[i]))
    if first_round is None:
        first_round = {i: (evals[i][0].mean() - g0.mean(), int(evals[i][1].sum() - p0.sum()), evals[i][0].min()) for i in free}
    g1, p1 = evals[best]
    chosen.append(best)
    for j, st in enumerate(states):                       # commit the plant
        t = cand_t[best][j]
        st.green = np.where(t < st.tmin, True, st.green)
        st.tmin = np.minimum(st.tmin, t)
        st.los += cand_los[best, j]
    log.append(dict(round=rnd, x=round(cands[best]['x'], 2), y=round(cands[best]['y'], 2), on=cands[best]['on'],
                    seats_pass=f"{int(p0.sum())} -> {int(p1.sum())}", min_gvi=f"{g0.min():.1f} -> {g1.min():.1f}",
                    mean_gvi=f"{g0.mean():.1f} -> {g1.mean():.1f}"))
    g0, p0 = g1, p1
if not chosen:
    print('No plant needed or none could be added.')
print(f"Added {len(chosen)} plant(s).")

## Step 7: Ranked placements

The plants in the order the search chose them (each row shows what that plant added on top of the earlier ones). `on` says whether it stands on the floor or on a piece of furniture. Coordinates are in meters in the same system as your CSV files.

In [ ]:
placements = pd.DataFrame(log)
display(placements)
new_objs = [new_plant(cands[i], f'new plant {k + 1}') for k, i in enumerate(chosen)]
objects_after = objects + new_objs
after = audit(seats, objects_after, params)
sa = share_passing(after, params)
print(f"After:  {sa['passed']}/{sa['seats']} seats pass ({sa['share_pct']}%), needed {sa['needed']}; "
      f"{'meets' if sa['meets'] else 'does not meet'} {params['threshold_pct']}%")
print(f"Before: {sb['passed']}/{sb['seats']} ({sb['share_pct']}%)  |  min GVI {before.gvi_pct.min():.1f}% -> {after.gvi_pct.min():.1f}%,"
      f" mean GVI {before.gvi_pct.mean():.1f}% -> {after.gvi_pct.mean():.1f}%")

## Step 7b: Seats passing and mean GVI after each plant

The curve shows what each added plant did to the number of seats that pass (left axis, with the threshold as a dashed line) and to the mean GVI (right axis). A flat step means that the plant added green but no new passing seat.

In [ ]:
rounds = [0] + [int(r['round']) for r in log]
passes = [int(sb['passed'])] + [int(str(r['seats_pass']).split('->')[1]) for r in log]
means = [float(before.gvi_pct.mean())] + [float(str(r['mean_gvi']).split('->')[1]) for r in log]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(rounds, passes, 'o-', color='#1f77b4', lw=2, label='Seats passing')
ax.axhline(need, color='k', ls='--', lw=1)
ax.text(rounds[-1], need + 0.08, f"{params['threshold_pct']}% = {need} seats", ha='right', fontsize=9)
ax.set_xlabel('Plants added'); ax.set_ylabel('Seats passing (line of sight)')
ax.set_xticks(rounds); ax.set_ylim(0, len(states) + 0.5)
ax2 = ax.twinx(); ax2.plot(rounds, means, 's--', color='#2e8b3a', lw=1.5, label='Mean GVI (%)'); ax2.set_ylabel('Mean GVI (%)')
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc='lower right', fontsize=9)
ax.set_title(f'Goal: {GOAL}')
plt.show()

## Step 8: Before and after

Left: today. Right: with the new plants (orange). Seat wedges show the view angle (green = pass, red = fail).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_plan(before, objects, params, ax=axes[0], title=f"Before: {sb['passed']}/{sb['seats']} seats pass")
plot_plan(after, objects_after, params, ax=axes[1], title=f"After: {sa['passed']}/{sa['seats']} seats pass")
for k, o in enumerate(new_objs):
    axes[1].add_patch(Circle(o['c'][:2], o['r'], color='#ff7f0e', alpha=0.85, lw=0))
    axes[1].text(o['c'][0], o['c'][1], str(k + 1), ha='center', va='center', color='white', fontsize=10, weight='bold')
plt.tight_layout()
plt.show()

## Step 9: Where would one extra plant help most?

For every candidate position this shows what **one** plant placed there alone would do to the average GVI of all seats, starting from today's room. Darker = bigger gain. The table lists the ten best single positions (a ranking you can use even if you do not follow the step-by-step choice above).

In [ ]:
if first_round:
    ids = list(first_round)
    X = np.array([cands[i]['x'] for i in ids]); Y = np.array([cands[i]['y'] for i in ids])
    G = np.array([first_round[i][0] for i in ids])
    fig, ax = plt.subplots(figsize=(9, 7))
    plot_plan(before, objects, params, ax=ax, title='Gain in mean seat GVI from one extra plant (percentage points)')
    sc = ax.scatter(X, Y, c=G, cmap='Oranges', s=90, edgecolor='k', linewidth=0.3, zorder=5)
    plt.colorbar(sc, ax=ax, shrink=0.7, label='mean GVI gain (points)')
    plt.show()
    top = sorted(ids, key=lambda i: (first_round[i][1], first_round[i][0]), reverse=True)[:10]
    display(pd.DataFrame([dict(x=round(cands[i]['x'], 2), y=round(cands[i]['y'], 2), on=cands[i]['on'],
                               extra_seats_pass=first_round[i][1], mean_gvi_gain=round(first_round[i][0], 2),
                               min_gvi_after=round(first_round[i][2], 1)) for i in top]))

## Step 10: Do the new plants hide the outdoor view?

A plant standing in front of a window can block the view of the lawn or trees outside. If your model has outdoor nature (objects tagged `exterior`), this table counts, for every seat, the rays that reach it before and after (the WELL exterior-view route from the Module 1 notebook) and flags any seat that loses its view. The optimizer does not take this into account when it chooses, so check it here.

In [ ]:
def exterior_rays(objs, seat):
    o = np.array([seat['x'], seat['y'], seat.get('eye_height', params['eye_height'])], float)
    D, _ = view_rays(seat['facing_deg'], params)
    ts = np.stack([np.full(len(D), np.inf) if ob['tag'] == 'glass' else hit_t(o, D, ob) for ob in objs], axis=1)
    first = np.array([ob['tag'] for ob in objs] + ['none'])[np.where(np.isfinite(ts.min(axis=1)), ts.argmin(axis=1), len(objs))]
    return int((first == 'exterior').sum())

if any(ob['tag'] == 'exterior' for ob in objects):
    ext_tab = pd.DataFrame([dict(seat_id=s['id'], ext_rays_before=exterior_rays(objects, s), ext_rays_after=exterior_rays(objects_after, s))
                            for s in seats])
    mr = params['min_clear_rays']
    ext_tab['view_before'] = ext_tab.ext_rays_before >= mr
    ext_tab['view_after'] = ext_tab.ext_rays_after >= mr
    display(ext_tab)
    lost = ext_tab[ext_tab.view_before & ~ext_tab.view_after]
    print(f"Seats with an exterior view: {int(ext_tab.view_before.sum())} before, {int(ext_tab.view_after.sum())} after.")
    print('Seats that LOSE their exterior view:', list(lost.seat_id) if len(lost) else 'none')
else:
    print("No outdoor nature (tag 'exterior') in the model, so there is nothing to check.")

## Step 11: Check against Module 1

The numbers above come from the fast search. Here every seat is scored again from scratch with the full Module 1 audit on the room *with* the new plants. If the two disagree, something is wrong and the notebook says so.

In [ ]:
full = audit(seats, objects_after, params)
g_fast, p_fast = current()
ok_gvi = np.allclose(full.gvi_pct.values, np.round(g_fast, 1), atol=0.11)
ok_pass = (full.los_pass.values == p_fast).all()
print('GVI matches the full audit:', ok_gvi, '| pass/fail matches:', ok_pass)
assert ok_gvi and ok_pass, 'Fast search and full audit disagree'
display(pd.DataFrame(dict(seat_id=before.seat_id, gvi_before=before.gvi_pct, gvi_after=after.gvi_pct,
                          pass_before=before.los_pass, pass_after=after.los_pass)))

## Step 12: Save the results

Writes the ranked placements, the seat table before and after, and `objects_with_new_plants.csv`, the same format as `objects.csv` with the new plants added, so you can load it back into Module 1 or model the plants in Rhino. The last cell downloads them (in Colab).

In [ ]:
import os
OUT_DIR = '/content/module3_results' if os.path.isdir('/content') else 'module3_results'
os.makedirs(OUT_DIR, exist_ok=True)
placements.to_csv(f'{OUT_DIR}/placements.csv', index=False)
before.merge(after[['seat_id', 'gvi_pct', 'los_pass']], on='seat_id', suffixes=('_before', '_after')).to_csv(f'{OUT_DIR}/seats_before_after.csv', index=False)
rows = []
for o in objects_after:
    if o['type'] == 'box':
        rows.append(dict(name=o['name'], type='box', tag=o['tag'], xmin=o['mn'][0], ymin=o['mn'][1], zmin=o['mn'][2],
                         xmax=o['mx'][0], ymax=o['mx'][1], zmax=o['mx'][2]))
    else:
        rows.append(dict(name=o['name'], type='sphere', tag=o['tag'], cx=o['c'][0], cy=o['c'][1], cz=o['c'][2], r=o['r']))
pd.DataFrame(rows).to_csv(f'{OUT_DIR}/objects_with_new_plants.csv', index=False)
fig.savefig(f'{OUT_DIR}/before_after.png', dpi=150, bbox_inches='tight') if 'fig' in globals() else None
pd.DataFrame([dict(goal=GOAL, size=PLANT_SIZE, max_plants=MAX_PLANTS, grid_step=GRID_STEP, seat_clearance=SEAT_CLEARANCE,
                   plant_gap=PLANT_GAP, put_on_top_of=PUT_ON_TOP_OF, keep_out=KEEP_OUT, **params)]).to_csv(f'{OUT_DIR}/settings_log.csv', index=False)
print(sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    import shutil
    shutil.make_archive('/content/module3_results', 'zip', OUT_DIR)
    files.download('/content/module3_results.zip')
except Exception as e:
    print('Not in Colab, files are in', OUT_DIR)